# G10 — Notebook 02 : Analyse des résultats P02

**Problématique** : Comment le weight decay et le dropout affectent-ils la généralisation de CamemBERT sur Allociné ?  
**Ce notebook** : Chargement des résultats Optuna + Visualisations

> ⚠️ Lancer ce notebook **après** `poetry run python -m src.optimization --mode both`

---

In [3]:
import sys, pickle, json
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from src.config import RESULTS_DIR, FIGURES_DIR
from src.visualization import (
    plot_generalization_heatmap,
    plot_convergence_curves,
    plot_optuna_importance,
    plot_sharpness_vs_f1,
)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Imports OK')

Imports OK


## 1. Chargement des résultats Optuna

In [4]:
# Charger l'étude Optuna
study_path = RESULTS_DIR / 'optuna_study.pkl'
with open(study_path, 'rb') as f:
    study = pickle.load(f)

print(f'Trials complétés : {len(study.trials)}')
print(f'Meilleur F1 val  : {study.best_value:.4f}')
print(f'Meilleurs params : {study.best_params}')

Trials complétés : 20
Meilleur F1 val  : 0.9533
Meilleurs params : {'weight_decay': 0.001, 'dropout': 0.0, 'learning_rate': 0.00013513720151519395}


In [5]:
# DataFrame des trials
df_trials = study.trials_dataframe()
df_trials = df_trials[df_trials['state'] == 'COMPLETE'].sort_values('value', ascending=False)
df_trials.head(10)

,number,value,datetime_start,datetime_complete,duration,params_dropout,params_learning_rate,params_weight_decay,user_attrs_gap,user_attrs_gap_pct,user_attrs_total_time_s,user_attrs_train_f1,state
17,17,0.953331,2026-03-06 18:36:36.565522,2026-03-06 19:02:01.827049,0 days 00:25:25.261527,0.0,0.000135,0.00100,0.002668,0.279084,1514.966223,0.955999,COMPLETE
12,12,0.946657,2026-03-05 14:18:49.585993,2026-03-05 14:38:05.006825,0 days 00:19:15.420832,0.0,0.000102,0.00100,0.001313,0.138490,1150.883293,0.947970,COMPLETE
14,14,0.939997,2026-03-05 14:55:35.123613,2026-03-05 18:13:10.962859,0 days 03:17:35.839246,0.0,0.000085,0.00100,0.006001,0.634327,6875.712592,0.945998,COMPLETE
18,18,0.939997,2026-03-06 19:02:02.282770,2026-03-06 19:41:21.612069,0 days 00:39:19.329299,0.1,0.000142,0.00100,-0.005998,-0.642141,2351.804545,0.934000,COMPLETE
13,13,0.933321,2026-03-05 14:38:05.255142,2026-03-05 14:55:34.900314,0 days 00:17:29.645172,0.0,0.000102,0.00100,0.028666,2.979893,1043.467082,0.961988,COMPLETE
16,16,0.926663,2026-03-06 18:06:34.551503,2026-03-06 18:36:36.199858,0 days 00:30:01.648355,0.0,0.000053,0.00001,0.021336,2.250610,1762.797395,0.947999,COMPLETE
15,15,0.926637,2026-03-05 18:13:11.354991,2026-03-06 09:41:15.380588,0 days 15:28:04.025597,0.1,0.000067,0.00100,0.005335,0.572492,55622.463191,0.931973,COMPLETE
6,6,0.919643,2026-03-05 11:48:55.939989,2026-03-05 12:19:14.157014,0 days 00:30:18.217025,0.0,0.000308,0.00100,0.040357,4.203805,1805.930923,0.959999,COMPLETE
0,0,0.913329,2026-03-05 09:08:18.860533,2026-03-05 09:37:19.134149,0 days 00:29:00.273616,0.0,0.000218,0.00010,0.036646,3.857604,1738.429643,0.949976,COMPLETE
10,10,0.913299,2026-03-05 13:36:54.169649,2026-03-05 14:02:46.867541,0 days 00:25:52.697892,0.0,0.000414,0.00100,0.022599,2.414664,1513.776664,0.935897,COMPLETE


## 2. Heatmap Gap de Généralisation × Weight Decay × Dropout

In [7]:
# Charger les résultats de la grille déterministe
grid_path = RESULTS_DIR / 'optuna_trials.csv'
df_grid = pd.read_csv(grid_path)
print(df_grid.to_string(index=False))

 number    value             datetime_start          datetime_complete               duration  params_dropout  params_learning_rate  params_weight_decay  user_attrs_gap  user_attrs_gap_pct  user_attrs_total_time_s  user_attrs_train_f1    state
      0 0.913329 2026-03-05 09:08:18.860533 2026-03-05 09:37:19.134149 0 days 00:29:00.273616             0.0              0.000218              0.00010        0.036646            3.857604              1738.429643             0.949976 COMPLETE
      1 0.644756 2026-03-05 09:37:19.573422 2026-03-05 09:59:08.016630 0 days 00:21:48.443208             0.0              0.000003              0.01000       -0.053714           -9.088024              1304.070302             0.591042 COMPLETE
      2 0.858178 2026-03-05 09:59:08.135496 2026-03-05 10:19:23.424951 0 days 00:20:15.289455             0.0              0.000010              0.00010       -0.070836           -8.996910              1211.165848             0.787342 COMPLETE
      3      NaN 2026-03

In [8]:
plot_generalization_heatmap(df_grid, save_path=FIGURES_DIR)

KeyError: 'weight_decay'

## 3. Importance des hyperparamètres (Optuna)

In [ ]:
plot_optuna_importance(study, save_path=FIGURES_DIR)

## 4. Convergence — Effet du Dropout

In [ ]:
# Charger les historiques sauvegardés (un par configuration)
import json

histories = {}
for dp in [0.0, 0.1, 0.3]:
    p = RESULTS_DIR / f'history_dp{dp:.1f}.json'
    if p.exists():
        with open(p) as f:
            histories[f'dropout={dp}'] = json.load(f)

if histories:
    plot_convergence_curves(histories, save_path=FIGURES_DIR)
else:
    print('Aucun historique trouvé. Lancez l\'optimisation en premier.')

## 5. Analyse des Minima — Sharpness vs F1

In [ ]:
# Charger les données de sharpness si disponibles
sharpness_path = RESULTS_DIR / 'sharpness_results.json'
if sharpness_path.exists():
    with open(sharpness_path) as f:
        sharpness_data = json.load(f)
    plot_sharpness_vs_f1(sharpness_data, save_path=FIGURES_DIR)
else:
    print('Données de sharpness non trouvées. Lancez visualization.py en premier.')

## 6. Tableau de synthèse — Top configurations

In [ ]:
# Top configurations par F1-val
top = df_grid.sort_values('val_f1', ascending=False).head(5)

print('\n=== TOP 5 CONFIGURATIONS (P02) ===')
print(f'{"weight_decay":>15} {"dropout":>10} {"F1_val":>10} {"F1_train":>10} {"Gap":>8}')
print('-' * 58)
for _, row in top.iterrows():
    print(f'{row["weight_decay"]:>15.0e} {row["dropout"]:>10.2f} '
          f'{row["val_f1"]:>10.4f} {row["train_f1"]:>10.4f} {row["gap"]:>8.4f}')

best = top.iloc[0]
print(f'\n→ Meilleure config : weight_decay={best["weight_decay"]:.0e}, dropout={best["dropout"]:.2f}')
print(f'  F1-score val : {best["val_f1"]:.4f} | Gap : {best["gap"]:.4f}')

## 7. Conclusions P02

Complétez cette section avec vos observations :

### Effet du weight decay
- ...

### Effet du dropout
- ...

### Interaction weight_decay × dropout
- ...

### Relation flatness des minima ↔ généralisation
- ...